# Training — Baseline CNN

Train a simple CNN baseline for comparison with the Multi-Head Expert model.

**Architecture** : 5 ConvBlocks + AdaptiveAvgPool + MLP head (GELU, no Sigmoid output)  
**Loss** : FocalAUCLoss (70% Focal γ=2.5 + 30% AUC pairwise)  
**Metrics saved** : ROC-AUC, PR-AUC, F1, Recall, Precision → `results/metrics/baseline_cnn.json`

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = os.environ.get('DATA_DIR', '/kaggle/input/rsna-breast-cancer-detection')
CODE_DIR = os.environ.get('CODE_DIR', '/kaggle/input/rsna-breast-cancer-code')
OUT_DIR  = os.environ.get('OUT_DIR',  '/kaggle/working/results')

sys.path.insert(0, CODE_DIR)
os.makedirs(f'{OUT_DIR}/metrics', exist_ok=True)
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'DATA_DIR : {DATA_DIR}')

In [ ]:
# ── Load & split data (patient-wise, no leakage) ──────────────────────────────
df = pd.read_csv(f'{DATA_DIR}/train.csv')
df['density'] = df['density'].fillna('B')
print(f'{len(df)} images — {df.cancer.mean()*100:.2f}% cancer — {df.patient_id.nunique()} patients')

# Patient-wise stratified split 70/15/15
from sklearn.model_selection import train_test_split

patients = df.groupby('patient_id')['cancer'].max().reset_index()
train_pats, tmp_pats = train_test_split(patients, test_size=0.3, stratify=patients['cancer'], random_state=42)
val_pats, test_pats  = train_test_split(tmp_pats,  test_size=0.5, stratify=tmp_pats['cancer'],  random_state=42)

df_train = df[df.patient_id.isin(train_pats.patient_id)].reset_index(drop=True)
df_val   = df[df.patient_id.isin(val_pats.patient_id)].reset_index(drop=True)
df_test  = df[df.patient_id.isin(test_pats.patient_id)].reset_index(drop=True)

print(f'Train: {len(df_train)} images ({df_train.cancer.mean()*100:.1f}% cancer)')
print(f'Val  : {len(df_val)} images ({df_val.cancer.mean()*100:.1f}% cancer)')
print(f'Test : {len(df_test)} images ({df_test.cancer.mean()*100:.1f}% cancer)')

In [ ]:
# ── Dataset & DataLoaders ─────────────────────────────────────────────────────
from torch.utils.data import DataLoader
from models.dataset import MammographyDataset, BalancedPatientSampler

IMAGES_DIR  = f'{DATA_DIR}/train_images'
TARGET_SIZE = (512, 512)
BATCH_SIZE  = 16

ds_train = MammographyDataset(df_train, IMAGES_DIR, target_size=TARGET_SIZE,
                               mode='train', preprocess_mode='full')
ds_val   = MammographyDataset(df_val,   IMAGES_DIR, target_size=TARGET_SIZE,
                               mode='val',   preprocess_mode='full')
ds_test  = MammographyDataset(df_test,  IMAGES_DIR, target_size=TARGET_SIZE,
                               mode='test',  preprocess_mode='full')

# Balanced sampler pour le train
train_labels = df_train['cancer'].values
train_pids   = df_train['patient_id'].values
sampler = BalancedPatientSampler(train_pids, train_labels, pos_weight=13.7)

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=4, pin_memory=True)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=4, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
from models.baseline_cnn import BaselineCNN

model = BaselineCNN(in_channels=1, dropout=0.4).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'BaselineCNN — {n_params:,} trainable parameters')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from models.trainer import Trainer

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    lr=3e-4,
    n_epochs=30,
    patience=7,
    checkpoint_dir=f'{OUT_DIR}/checkpoints',
    pos_weight=13.7,
    use_amp=(DEVICE == 'cuda')
)

history = trainer.train()
print('Entraînement terminé.')

In [ ]:
# ── Courbes d'entraînement ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Baseline CNN — Training curves', fontsize=13)

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['val_auroc'])
axes[1].set_title('Val ROC-AUC')
axes[1].axhline(0.66, color='red', linestyle='--', label='v1 baseline (0.66)')
axes[1].legend()

axes[2].plot(history['val_f1'])
axes[2].set_title('Val F1')

plt.tight_layout()
fig_path = f'{OUT_DIR}/figures/baseline_cnn_training.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Évaluation sur le test set ────────────────────────────────────────────────
from sklearn.metrics import roc_curve, f1_score, precision_score, recall_score

model.eval()
all_probs, all_labels, all_pids = [], [], []

with torch.no_grad():
    for imgs, labels, pids in test_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
        all_pids.extend(pids.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
all_pids   = np.array(all_pids)

# Patient-level aggregation (max over images)
from models.dataset import MammographyDataset
pat_probs, pat_ids = MammographyDataset.patient_level_aggregate(all_probs, all_pids, method='max')
pat_labels = np.array([all_labels[all_pids == pid].max() for pid in pat_ids])

auroc    = roc_auc_score(pat_labels, pat_probs)
prauc    = average_precision_score(pat_labels, pat_probs)

# Threshold optimal (Youden)
fpr, tpr, thresholds = roc_curve(pat_labels, pat_probs)
best_thr = thresholds[np.argmax(tpr - fpr)]
preds    = (pat_probs >= best_thr).astype(int)

f1        = f1_score(pat_labels, preds, zero_division=0)
recall    = recall_score(pat_labels, preds, zero_division=0)
precision = precision_score(pat_labels, preds, zero_division=0)

print(f'\n=== Baseline CNN — Test set (patient-level) ===')
print(f'ROC-AUC   : {auroc:.4f}')
print(f'PR-AUC    : {prauc:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Threshold : {best_thr:.4f}')

In [ ]:
# ── Courbe ROC ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, label=f'Baseline CNN (AUC={auroc:.3f})')
ax.plot([0,1],[0,1], 'k--')
ax.axvline(fpr[np.argmax(tpr-fpr)], color='red', linestyle=':', alpha=0.5, label=f'Best threshold={best_thr:.2f}')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve — Baseline CNN (patient-level)')
ax.legend()
plt.tight_layout()
roc_path = f'{OUT_DIR}/figures/baseline_cnn_roc.png'
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sauvegarde métriques JSON ─────────────────────────────────────────────────
metrics = {
    'model': 'BaselineCNN',
    'n_params': n_params,
    'n_epochs_trained': len(history['val_auroc']),
    'best_val_auroc': float(max(history['val_auroc'])),
    'test_patient_level': {
        'roc_auc':   round(float(auroc), 4),
        'pr_auc':    round(float(prauc), 4),
        'f1':        round(float(f1), 4),
        'recall':    round(float(recall), 4),
        'precision': round(float(precision), 4),
        'threshold': round(float(best_thr), 4)
    },
    'preprocess_mode': 'full',
    'target_size': list(TARGET_SIZE)
}

json_path = f'{OUT_DIR}/metrics/baseline_cnn.json'
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('Métriques sauvegardées :')
print(json.dumps(metrics, indent=2))